In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from utils import generate_discrete_variates, normal_variables, get_num_days_in_month
from random_variate_generator import RandomVariateGenerator

BLUE   = '#2a78d6'
RED    = '#e34948'
SEED = 42
NUM_CUSTOMERS = 5
NUM_TRANSACTIONS = 50
FRAUD_MEAN = 140
FRAUD_STANDARD_DEVIATION = 15
FRAUD_PERCENT_BETWEEN_1_AND_4 = 0.113 # 0.113 from given data
FRAUD_PERCENT_NOT_BETWEEN_1_AND_4 = 0.0245 #0.0245 from given data
urv = RandomVariateGenerator(SEED)

In [ ]:
df = pd.read_csv('./data/credit_card_fraud_detection_2026.csv')

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Hourly fraud rate
hourly = df.groupby('transaction_hour').agg(
    total=('is_fraud', 'count'),
    fraud=('is_fraud', 'sum')
).reset_index()
hourly['fraud_rate'] = hourly['fraud'] / hourly['total'] * 100
baseline_rate = df['is_fraud'].mean() * 100

axes[0].bar(hourly['transaction_hour'], hourly['fraud_rate'],
              color=[RED if r > baseline_rate * 1.5 else BLUE
                     for r in hourly['fraud_rate']], edgecolor='none')
axes[0].axhline(baseline_rate, color='black', linewidth=1.2,
                   linestyle='--', label=f'Baseline {baseline_rate:.2f}%')
axes[0].set_title('Fraud Rate by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour (0 = Midnight)')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].set_xticks(range(0, 24, 2))

# Simplified Piecewise Uniform Distribution for Fraud Rate ---
bins_top = [1,4,5,23]
fraud_rates = hourly.set_index('transaction_hour')['fraud_rate'].reindex(range(24), fill_value=0)

for i in range(len(bins_top) - 1):
    start, end = bins_top[i], bins_top[i+1]
    
    block_slice = fraud_rates.loc[start:end-1]
    block_height = block_slice.max()
    
    label_str = f"Uniform [{start}-{end}h] | Max Height: {block_height:.2f}%"
    
    axes[0].plot([start, end], [block_height, block_height], color='darkorange', linewidth=3, label=label_str)
    
    axes[0].fill_between([start, end], 0, [block_height, block_height], color='orange', alpha=0.1)

axes[0].legend(loc='upper right', fontsize=8)


# Transaction volume by hour
axes[1].bar(hourly['transaction_hour'], hourly['total'],
              color=BLUE, edgecolor='none', alpha=0.7, label='Legitimate')
fraud_hourly = df[df.is_fraud==1].groupby('transaction_hour').size()
axes[1].bar(fraud_hourly.index, fraud_hourly.values,
              color=RED, edgecolor='none', label='Fraud')
axes[1].set_title('Transaction Volume by Hour', fontweight='bold')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Transaction Count')
axes[1].set_xticks(range(0, 24, 2))

hour_counts = hourly.set_index('transaction_hour')['total'].reindex(range(24), fill_value=0)

# Simplified Piecewise Uniform Distribution for Transaction Volume ---
block_area = 0
sample_area = []
bins = [0, 6, 9, 14, 17, 21, 24]

for i in range(len(bins) - 1):
    start, end = bins[i], bins[i+1]
    
    block_slice = hour_counts.loc[start:end-1]
    block_height = block_slice.max()
    sample_area.append(block_height * (end - start))
    block_area += sample_area[-1]
    
    label_str = f"Uniform [{start}-{end}h] | Height: {block_height:,.0f} | Area: {sample_area[-1]:,.0f}"
    
    axes[1].plot([start, end], [block_height, block_height], color='darkblue', linewidth=3, label=label_str)
    
    axes[1].fill_between([start, end], 0, [block_height, block_height], color='blue', alpha=0.1)

axes[1].legend(loc='upper left', fontsize=8)
print(f"Total block area: {block_area:,.0f}")


In [ ]:
rx = {}
for i in range(len(sample_area)):
    rx[bins[i]] = sample_area[i] / block_area
    print(f"Block starting at {bins[i]}: Sample Area = {sample_area[i]}")
print()

list(rx.values())

In [ ]:
from datetime import datetime

def transaction_time(probabilities, count = 1):
    # 2026-01-01 02:28:51
    datetimes = []
    for _ in range(count):
        month = urv.uniform_int(1, 12)
        max_day = get_num_days_in_month(month)
        day = urv.uniform_int(1, max_day)
        minute, second = urv.uniform_int(0, 59, size = 2)

        u =  urv.uniform()
        if u < probabilities[0]:
            hour = int(urv.uniform_int(0, 6))       # 0–5
        elif u < sum(probabilities[:2]):
            hour = int(urv.uniform_int(6, 9))       # 6–8
        elif u < sum(probabilities[:3]):
            hour = int(urv.uniform_int(9, 14))      # 9–13
        elif u < sum(probabilities[:4]):
            hour = int(urv.uniform_int(14, 17))     # 14–16
        elif u < sum(probabilities[:5]):
            hour = int(urv.uniform_int(17, 21))     # 17–20
        else:
            hour = int(urv.uniform_int(21, 24))     # 21–23

        # Format it as a datetime object datetime.datetime(year, month,day, hour, minute, second)
        datetimes.append(datetime(2025, month, day, hour, minute, second))
    return datetimes
time =  sorted(transaction_time(list(rx.values()), count = 10))

In [ ]:
def round_list(my_list):
    for i, val in enumerate(my_list):
        my_list[i] = round(val, 2)
    return my_list

In [ ]:
people_data = pd.DataFrame({
    'customer_id': range(1, NUM_CUSTOMERS + 1),
    # 'average_count_transactions': generate_discrete_variates(values=[1,2,3,4,5], probabilities=[0.05, 0.2, 0.4, 0.2, 0.15], count=NUM_CUSTOMERS, generator=urv),
    # 'standard_deviation_count_transactions': generate_discrete_variates(values=[1,2], probabilities=[0.4, 0.6], count=NUM_CUSTOMERS, generator=urv),
    'average_amount_per_transaction': round_list(normal_variables(generator=urv, mean=114, stddev=50, lower=0.0, count=NUM_CUSTOMERS)),
    'standard_deviation_amount_per_transaction': round_list(normal_variables(generator=urv, mean=40, stddev=5, lower=0.0, count=NUM_CUSTOMERS)),
})
people_data.to_csv('data/people_data.csv', index=False)

In [ ]:
transaction_table = pd.DataFrame({
    'customer_id': generate_discrete_variates(values=range(1, NUM_CUSTOMERS + 1), count=NUM_TRANSACTIONS, generator=urv),
    'transaction_time': sorted(transaction_time(list(rx.values()), count = NUM_TRANSACTIONS)),
})

In [ ]:
transaction_table['is_fraud'] = [
    int(urv.uniform() < (FRAUD_PERCENT_BETWEEN_1_AND_4 if 1 <= time.hour <= 4 else FRAUD_PERCENT_NOT_BETWEEN_1_AND_4))
    for time in transaction_table['transaction_time']
]
transaction_table['transaction_id'] = range(1, len(transaction_table) + 1)

In [ ]:
transaction_amount = []
for i in range(len(transaction_table)):
    if transaction_table.loc[i, 'is_fraud'] == 1:
        cust_average = FRAUD_MEAN
        cust_sd = FRAUD_STANDARD_DEVIATION
    else:
        cust_average = people_data.loc[people_data['customer_id'] == transaction_table.loc[i, 'customer_id'], 'average_amount_per_transaction'].values[0]
        cust_sd = people_data.loc[people_data['customer_id'] == transaction_table.loc[i, 'customer_id'], 'standard_deviation_amount_per_transaction'].values[0]
    amount = round(urv.normal(cust_average, cust_sd), 2)
    while amount <= 0:
        print(f"Generated non-positive amount for customer {transaction_table.loc[i, 'customer_id']}, retrying... Generated amount: {amount}")
        amount = round(urv.normal(cust_average, cust_sd), 2)
    transaction_amount.append(amount)
transaction_table['amount'] = transaction_amount

In [ ]:
transaction_table.to_csv('data/transaction_table.csv', index=False)